In [2]:
# 1. Import Necessary Libraries
#==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import statsmodels
import scipy
import os

In [3]:
#set default styles so the notebook looks clean
#==============================================================================
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [4]:
#Download the latest version of the dataset from kaggle
#==============================================================================
print("Loading dataset...")

dataset_path = kagglehub.dataset_download("pratyushpuri/financial-news-market-events-dataset-2025")
print("Files saved at path:", dataset_path)

#Find the csv file in the downloaded directory 
csv_files = [file for file in os.listdir(dataset_path) if file.endswith('csv')]

if not csv_files:
    raise FileNotFoundError("No CSV file found in the downloaded Kaggle dataset.")

#Assuming the data is in the first csv found
file_path = os.path.join(dataset_path, csv_files[0])
df = pd.read_csv(file_path)

#Convert string object to datetime format 
df['Date'] = pd.to_datetime(df['Date'])

#Create new columns for Weekday, Month, and Week number 
df['Weekday'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month
df['Week_number'] = df['Date'].dt.isocalendar().week

print(df[['Date','Weekday','Month', 'Week_number']].head(10))

Loading dataset...
Files saved at path: /Users/tajsharma/.cache/kagglehub/datasets/pratyushpuri/financial-news-market-events-dataset-2025/versions/1
        Date    Weekday  Month  Week_number
0 2025-05-21  Wednesday      5           21
1 2025-05-18     Sunday      5           20
2 2025-06-25  Wednesday      6           26
3 2025-07-21     Monday      7           30
4 2025-07-23  Wednesday      7           30
5 2025-03-18    Tuesday      3           12
6 2025-03-02     Sunday      3            9
7 2025-04-05   Saturday      4           14
8 2025-04-08    Tuesday      4           15
9 2025-06-18  Wednesday      6           25


# MECE Framework & Hypotheses

## Research Question
**What type of news moves the market?**

---

## Analytical Branches
| # | Branch | Dimension | Key Column(s) |
|---|--------|-----------|---------------|
| 1 | What happened? | Event type | `Market_Event` |
| 2 | Where it happened? | Sector | `Sector` |
| 3 | Who was involved? | Company | `Related_Company` |
| 4 | How was it reported? | Tone & source | `Sentiment`, `Source` |
| 5 | When it happened? | Timing | `Date`, `Weekday`, `Month` |
| 6 | How significant was it perceived? | Magnitude & market | `Impact_Level`, `Market_Index` |

---

## Hypotheses

### Branch 1 — What happened? (`Market_Event`)

**H1a:** Policy/regulatory events will cause higher volatility in market events because most traders take these into consideration as these events directly impact how companies can operate and it influences how they import/export services/goods.

**H1b:** Macro-economic indicators move slowly and are often priced in before official releases, so the actual announcement may cause less surprise and therefore less volatility.

---

### Branch 2 — Where it happened? (`Sector`)

**H2a:** Sectors will show clearly different levels of volatility. Tech (reliance on future earnings, rapid innovation, new players), Energy (gas/oil price fluctuations), Materials (manufacturing laws, supply chain), and Pharmaceuticals (depends on patents and approvals) will be significantly more volatile than Consumer Goods (constant demand), Utilities (essential services, lower price fluctuations), and Real Estate (tangible assets, long-term value).

**H2b:** The spread between the most and least volatile sectors will be under 0.5 percentage points because the news events in this dataset are macro-level in nature — they affect entire markets, not individual industries. Sector-specific sensitivity exists but is muted by the broad scope of these events.

---

### Branch 3 — Who was involved? (`Related_Company`)

**H3a:** Financial institutions like JP Morgan will show disproportionately higher volatility during Central Bank Meetings and Interest Rate Changes compared to their overall average, while tech companies like Tesla or Apple will spike more during Regulatory Changes and Trade Tariff Announcements — because each company's core business model is exposed to different policy levers.

**H3b:** Companies that appear more frequently in the dataset will show lower average volatility per event because frequent coverage suggests routine news flow, while companies that appear rarely are only covered during exceptional events which tend to be more disruptive.

---

### Branch 4 — How was it reported? (`Sentiment`, `Source`)

**H4a:** Negative sentiment will drive more volatility because humans are more risk-averse — they are more likely to sell a stock on bad news ("the internet will not be needed") than to buy on good news ("a lot of growth potential"). This aligns with Kahneman's Prospect Theory: losses are felt roughly twice as strongly as equivalent gains.

**H4b:** News source will have minimal independent effect on volatility because all outlets are reporting on the same underlying events. Any differences in volatility by source will disappear once you control for event type — meaning the source's apparent effect is actually just a reflection of which events they choose to cover.

---

### Branch 5 — When it happened? (`Date`, `Weekday`, `Month`)

**H5a:** Mondays will show higher average volatility than other weekdays because news accumulates over the weekend and hits the market all at once. Fridays will show the second-highest volatility as traders rebalance positions before the weekend to reduce risk exposure.

**H5b:** Volatility will not be evenly distributed across the dataset's timeline — it will cluster into "regimes" where several weeks of elevated volatility are followed by calm periods. These clusters will coincide with periods where multiple high-impact event types occurred simultaneously.

**H5c:** Months that coincide with corporate earnings season (January, April, July, October) will show elevated volatility because earnings reports flood the market with new information simultaneously, forcing rapid repricing across sectors.

---

### Branch 6 — How significant was it perceived? (`Impact_Level`, `Market_Index`)

**H6a:** Events labeled "High" impact will show at least 50% more average volatility than "Low" impact events. If this relationship is weak, it suggests the Impact_Level label in the dataset is poorly assigned and not a reliable feature.

**H6b:** Different indices will show different baseline volatility levels. A tech-heavy index like NASDAQ will show higher volatility on Technology and IPO events, while a broad index like the S&P 500 will react more evenly across event types because it is diversified.

**H6c:** High impact + negative sentiment events will show disproportionately higher volatility than what you'd expect from adding the two effects separately. In other words, the combination is worse than the sum of its parts.


In [9]:
#Quick summary of raw dataset
print(df.info())

categorical_cols = df.select_dtypes(include=['object', 'category']).columns
numerical_cols = df.select_dtypes(include= ['number']).columns

for col in categorical_cols:
    print(f"--------- Column name: {col} ---------")
    print(df[col].value_counts())

for col in numerical_cols:
    print(f"--------- Column name: {col} ---------")
    print(df[col].describe())

null_percentage = df.isnull().mean() * 100

print(null_percentage.sort_values(ascending=False))

<class 'pandas.DataFrame'>
RangeIndex: 3024 entries, 0 to 3023
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Date                  3024 non-null   datetime64[us]
 1   Headline              2876 non-null   str           
 2   Source                3024 non-null   str           
 3   Market_Event          3024 non-null   str           
 4   Market_Index          3024 non-null   str           
 5   Index_Change_Percent  2863 non-null   float64       
 6   Trading_Volume        3024 non-null   float64       
 7   Sentiment             2853 non-null   str           
 8   Sector                3024 non-null   str           
 9   Impact_Level          3024 non-null   str           
 10  Related_Company       3024 non-null   str           
 11  News_Url              2871 non-null   str           
 12  Weekday               3024 non-null   str           
 13  Month                 3024 no

/var/folders/k5/1ys4jvv95zv528gdmtsrb7s80000gn/T/ipykernel_87312/3076122661.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object', 'category']).columns


In [ ]:
#Clean the data

#Drop all null values from the Sentiment and Index Change Percentage columns
df = df.dropna(subset=["Sentiment", "Index_Change_Percent"])
df.info()

#Check to see if there any duplicated rows
duplicate_rows_count = df.duplicated().sum()
print(f'There are {duplicate_rows_count} duplicated rows in the dataset.')


<class 'pandas.DataFrame'>
Index: 2704 entries, 2 to 3023
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Date                  2704 non-null   datetime64[us]
 1   Headline              2570 non-null   str           
 2   Source                2704 non-null   str           
 3   Market_Event          2704 non-null   str           
 4   Market_Index          2704 non-null   str           
 5   Index_Change_Percent  2704 non-null   float64       
 6   Trading_Volume        2704 non-null   float64       
 7   Sentiment             2704 non-null   str           
 8   Sector                2704 non-null   str           
 9   Impact_Level          2704 non-null   str           
 10  Related_Company       2704 non-null   str           
 11  News_Url              2570 non-null   str           
 12  Weekday               2704 non-null   str           
 13  Month                 2704 non-nul